# EDA 008: User Prior Aggregates (Leakage-Safe)

Goal: build reviewer-history features for each review using only records with `timestamp_created` strictly earlier than the current row.

In [1]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

TRAIN_PATH = Path("../../data/interim/steam_reviews_cleaned_english_train.parquet")
VAL_PATH = Path("../../data/interim/steam_reviews_cleaned_english_val.parquet")
TEST_PATH = Path("../../data/interim/steam_reviews_cleaned_english_test.parquet")
OUT_DIR = Path("../../data/interim/features")
OUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_COLS = [
    "review_id",
    "author.steamid",
    "timestamp_created",
    "recommended",
    "votes_helpful",
    "review_length_chars",
]

In [2]:
def _load_split(path: Path, split_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_parquet(path)
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{split_name} missing required columns: {missing}")
    out = df[REQUIRED_COLS].copy()
    out["split"] = split_name
    return out


train = _load_split(TRAIN_PATH, "train")
val = _load_split(VAL_PATH, "val")
test = _load_split(TEST_PATH, "test")

full = pd.concat([train, val, test], ignore_index=True)
full["timestamp_created"] = full["timestamp_created"].astype("int64")
full["recommended_int"] = full["recommended"].astype("int64")
full["review_length_chars"] = full["review_length_chars"].fillna(0).astype("float64")
full["votes_helpful"] = full["votes_helpful"].fillna(0).astype("float64")

full.shape

(9160492, 8)

In [3]:
def build_user_priors_strict(df: pd.DataFrame) -> pd.DataFrame:
    keys = ["author.steamid", "timestamp_created"]
    ts_agg = (
        df.groupby(keys, as_index=False)
        .agg(
            ts_count=("review_id", "size"),
            ts_rec_sum=("recommended_int", "sum"),
            ts_help_sum=("votes_helpful", "sum"),
            ts_len_sum=("review_length_chars", "sum"),
        )
        .sort_values(keys)
    )

    g = ts_agg.groupby("author.steamid", sort=False)
    ts_agg["user_prior_review_count"] = g["ts_count"].cumsum()
    ts_agg["user_prior_rec_sum"] = g["ts_rec_sum"].cumsum()
    ts_agg["user_prior_help_sum"] = g["ts_help_sum"].cumsum()
    ts_agg["user_prior_len_sum"] = g["ts_len_sum"].cumsum()
    ts_agg["user_prior_review_count"] = g["user_prior_review_count"].shift(1).fillna(0)
    ts_agg["user_prior_rec_sum"] = g["user_prior_rec_sum"].shift(1).fillna(0)
    ts_agg["user_prior_help_sum"] = g["user_prior_help_sum"].shift(1).fillna(0)
    ts_agg["user_prior_len_sum"] = g["user_prior_len_sum"].shift(1).fillna(0)
    ts_agg["user_prev_timestamp"] = g["timestamp_created"].shift(1)

    denom = ts_agg["user_prior_review_count"].replace(0, np.nan)
    ts_agg["user_prior_recommend_rate"] = (ts_agg["user_prior_rec_sum"] / denom).fillna(0.0)
    ts_agg["user_prior_mean_votes_helpful"] = (ts_agg["user_prior_help_sum"] / denom).fillna(0.0)
    ts_agg["user_prior_mean_review_len"] = (ts_agg["user_prior_len_sum"] / denom).fillna(0.0)
    ts_agg["user_seconds_since_last_review"] = (
        ts_agg["timestamp_created"] - ts_agg["user_prev_timestamp"]
    ).fillna(-1)

    merge_cols = [
        "author.steamid",
        "timestamp_created",
        "user_prior_review_count",
        "user_prior_recommend_rate",
        "user_prior_mean_votes_helpful",
        "user_prior_mean_review_len",
        "user_seconds_since_last_review",
    ]
    out = df.merge(ts_agg[merge_cols], on=["author.steamid", "timestamp_created"], how="left")
    out["user_prior_review_count"] = out["user_prior_review_count"].fillna(0).astype("int64")
    for c in [
        "user_prior_recommend_rate",
        "user_prior_mean_votes_helpful",
        "user_prior_mean_review_len",
        "user_seconds_since_last_review",
    ]:
        out[c] = out[c].fillna(0.0)
    return out


user_feat = build_user_priors_strict(full)
user_feat[["review_id", "split", "user_prior_review_count", "user_prior_recommend_rate"]].head()

,review_id,split,user_prior_review_count,user_prior_recommend_rate
0,85184605,train,0,0.0
1,85184171,train,0,0.0
2,85184064,train,1,1.0
3,85180436,train,0,0.0
4,85179753,train,0,0.0


In [4]:
# Leakage checks and spot checks
first_rows = (
    user_feat.sort_values(["author.steamid", "timestamp_created", "review_id"])
    .groupby("author.steamid", as_index=False)
    .head(1)
)
assert (first_rows["user_prior_review_count"] == 0).all(), "First user row should have zero prior count"

same_ts_counts = (
    user_feat.groupby(["author.steamid", "timestamp_created"])["user_prior_review_count"].nunique().max()
)
assert same_ts_counts == 1, "Rows with same timestamp must share identical strict-prior features"

sample = user_feat.sample(n=min(10, len(user_feat)), random_state=42)[
    [
        "review_id",
        "author.steamid",
        "timestamp_created",
        "user_prior_review_count",
        "user_prior_recommend_rate",
        "user_prior_mean_votes_helpful",
    ]
]
sample

,review_id,author.steamid,timestamp_created,user_prior_review_count,user_prior_recommend_rate,user_prior_mean_votes_helpful
7761957,30224160,76561198163679450,1488216657,0,0.0,0.0
6689153,19983304,76561198174926474,1451187376,2,1.0,0.0
7534727,72272304,76561198166342608,1594179623,3,1.0,0.0
2389987,41472153,76561197990179583,1523671182,1,0.0,6.0
4729440,68157709,76561198238128316,1588049905,2,1.0,0.0
2066135,77201864,76561199091747391,1602110866,1,1.0,0.0
4035568,70212575,76561198999731846,1591136228,0,0.0,0.0
8714972,39145619,76561198344180946,1515316293,0,0.0,0.0
1302006,17038685,76561198075995178,1436850690,1,1.0,0.0
2821539,65447474,76561198077646117,1584766383,0,0.0,0.0


In [7]:
# Only get users that have more than one review
user_counts = user_feat["author.steamid"].value_counts()
multi_review_users = user_counts[user_counts > 1].index
user_feat[user_feat["author.steamid"].isin(multi_review_users)].sort_values(
    ["author.steamid", "timestamp_created", "review_id"]
).head(15)

,review_id,author.steamid,timestamp_created,recommended,votes_helpful,review_length_chars,split,recommended_int,user_prior_review_count,user_prior_recommend_rate,user_prior_mean_votes_helpful,user_prior_mean_review_len,user_seconds_since_last_review
6394246,69928056,76561197960265745,1590649294,1,1.0,81.0,train,1,0,0.0,0.000000,0.0,-1.0
4839737,73534249,76561197960265745,1596147287,1,2.0,520.0,train,1,1,1.0,1.000000,81.0,5497993.0
7189905,27127273,76561197960265778,1479987803,1,1.0,11.0,val,1,0,0.0,0.000000,0.0,-1.0
2226281,67687984,76561197960265778,1587390881,1,0.0,251.0,train,1,1,1.0,1.000000,11.0,107403078.0
6214594,26702247,76561197960265781,1479599829,1,2.0,625.0,train,1,0,0.0,0.000000,0.0,-1.0
7947469,59242845,76561197960265781,1575495529,1,0.0,101.0,test,1,1,1.0,2.000000,625.0,95895700.0
6407478,65711325,76561197960265781,1585060343,1,0.0,432.0,train,1,2,1.0,1.000000,363.0,9564814.0
6745043,66985379,76561197960265781,1586470489,1,0.0,230.0,val,1,3,1.0,0.666667,386.0,1410146.0
8734477,30829038,76561197960265822,1490862846,0,0.0,99.0,test,0,0,0.0,0.000000,0.0,-1.0
8076761,40972738,76561197960265822,1521718770,1,0.0,144.0,test,1,1,0.0,0.000000,99.0,30855924.0


In [ ]:
# feature_cols = [
#     "review_id",
#     "user_prior_review_count",
#     "user_prior_recommend_rate",
#     "user_prior_mean_votes_helpful",
#     "user_prior_mean_review_len",
#     "user_seconds_since_last_review",
# ]

# for split_name in ["train", "val", "test"]:
#     split_out = user_feat.loc[user_feat["split"] == split_name, feature_cols].copy()
#     out_path = OUT_DIR / f"steam_reviews_{split_name}_user_prior_features.parquet"
#     split_out.to_parquet(out_path, index=False)
#     print(split_name, len(split_out), out_path)


train 6410755 ../../data/interim/features/steam_reviews_train_user_prior_features.parquet
val 1374737 ../../data/interim/features/steam_reviews_val_user_prior_features.parquet
test 1375000 ../../data/interim/features/steam_reviews_test_user_prior_features.parquet
